## 1. 路径配置与标签加载

In [ ]:
## 1. 路径配置与标签加载
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.patches import Rectangle, Patch
import h5py
import pandas as pd
from collections import Counter
from scipy.stats import entropy

# ============================================================================
# 路径配置 - 修改此处切换被试
# ============================================================================

SUBJECT_ID = "ODP_01_qhlazec"
SPLIT_TYPE = "test"

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")

if SPLIT_TYPE == "test":
    fold_candidates = list(BASE_RESULT_DIR.glob(f"*_test_{SUBJECT_ID}"))
    FOLD_DIR = fold_candidates[0] if len(fold_candidates) > 0 else BASE_RESULT_DIR / f"fold_01_test_{SUBJECT_ID}"
else:
    FOLD_DIR = BASE_RESULT_DIR / "fold_01_test_ODP_01_qhlazec"

DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")
DOWNSAMPLED_FILE = DATA_ROOT / f"{SUBJECT_ID}_downsampled.npz"
PRED_SOFTMAX_FILE = FOLD_DIR / "pred_3d" / f"{SPLIT_TYPE}_{SUBJECT_ID}_pred_softmax_3d.npz"
GT_3D_FILE = DATA_ROOT / "3d" / f"{SUBJECT_ID}_3d.npz"
LABEL_EXCEL = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")

OUTPUT_DIR = FOLD_DIR / "figs" / "confidence_overlay"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL_MPRAGE = 341
CHANNEL_QSM = 350

# ============================================================================
# 可视化参数
# ============================================================================

TAU = 0.4
N_CLASSES = 102
SLICE_AXIS = 'axial'
AUTO_SELECT_SLICES = True
MANUAL_SLICES = [30, 50, 70]
CONTOUR_LEVELS = [0.3]
CONTOUR_LINEWIDTH = 1.5

ZOOM_REGIONS = {
    "cortex_white": {"z": None, "x": None, "y": None, "hw": 20},
    "basal_ganglia": {"z": None, "x": None, "y": None, "hw": 20},
    "brainstem": {"z": None, "x": None, "y": None, "hw": 20}
}

print("✓ 路径配置完成")
print(f"  被试: {SUBJECT_ID}, 类型: {SPLIT_TYPE}")
print(f"  Fold: {FOLD_DIR.name}")
print(f"  输出: {OUTPUT_DIR}")

# ============================================================================
# 加载FreeSurfer标签映射
# ============================================================================

print("\n加载FreeSurfer标签映射...")
label_df = pd.read_excel(LABEL_EXCEL)
valid_labels = label_df[label_df['one_hot_loc_alex_label'] != '[]'].copy()
valid_labels['label_idx'] = valid_labels['one_hot_loc_alex_label'].astype(int)

LABEL_NAMES = {0: 'Background'}
LABEL_COLORS_RGB = {0: (0, 0, 0)}

for idx, row in valid_labels.iterrows():
    label_idx = int(row['label_idx'])
    if 1 <= label_idx <= N_CLASSES and label_idx not in LABEL_NAMES:
        LABEL_NAMES[label_idx] = row['tissue_name'].strip("'")
        LABEL_COLORS_RGB[label_idx] = (row['R']/255, row['G']/255, row['B']/255)

colors_list = [LABEL_COLORS_RGB.get(i, (0, 0, 0)) for i in range(N_CLASSES)]
cmap_freesurfer = mcolors.ListedColormap(colors_list)

print(f"✓ 加载了 {len(LABEL_NAMES)} 个标签")

In [ ]:
## 2. 加载数据并计算置信度与不确定性
# print("加载数据...")

# 预测概率
pred_data = np.load(PRED_SOFTMAX_FILE)
pred_softmax = pred_data['pred_softmax_3d']
print(f"  预测概率: {pred_softmax.shape}")

# Ground Truth
gt_data = np.load(GT_3D_FILE)
gt_proba = gt_data['proba_labels']
region_mask = gt_data['region_mask_lr']
print(f"  Ground Truth: {gt_proba.shape}")

# MPRAGE底图
if DOWNSAMPLED_FILE.exists():
    downsampled_data = np.load(DOWNSAMPLED_FILE)
    data_lr = downsampled_data['data_lr']
    anatomy_img = data_lr[..., CHANNEL_MPRAGE]
    anatomy_name = "MPRAGE"
    print(f"  MPRAGE底图: {anatomy_img.shape}")
else:
    anatomy_img = np.zeros(pred_softmax.shape[:3])
    anatomy_name = "No Anatomy"

# ============================================================================
# 计算置信度
# ============================================================================

print("\n计算置信度与不确定性...")

top2_indices = np.argsort(pred_softmax, axis=-1)[..., -2:]
top1_class = top2_indices[..., 1]
top2_class = top2_indices[..., 0]

top2_probs = np.take_along_axis(pred_softmax, top2_indices, axis=-1)
p1 = top2_probs[..., 1]
p2 = top2_probs[..., 0]

# 边际差 (margin)
margin = p1 - p2  # (Z, X, Y)
alpha_map = np.clip(margin / TAU, 0, 1)
alpha_map[region_mask == 0] = 0

# ============================================================================
# 计算熵图（不确定性）
# ============================================================================

# H(p) = -Σ_k p_k log(p_k)
# 使用scipy.stats.entropy，axis=-1计算每个体素的熵
epsilon = 1e-10  # 避免log(0)
pred_softmax_safe = np.clip(pred_softmax, epsilon, 1.0)

# entropy使用自然对数，转换为bits需要除以log(2)
entropy_map = entropy(pred_softmax_safe.T, axis=0).T  # scipy要求axis=0，所以转置
entropy_map = entropy_map / np.log(2)  # 转为bits

# 应用ROI掩码
entropy_map_masked = entropy_map.copy()
entropy_map_masked[region_mask == 0] = 0

# 统计
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

print(f"  边际差 (Margin, ROI内):")
print(f"    Mean={margin_roi.mean():.4f}, Median={np.median(margin_roi):.4f}")
print(f"    Range=[{margin_roi.min():.4f}, {margin_roi.max():.4f}]")

print(f"  熵 (Entropy, ROI内):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Median={np.median(entropy_roi):.4f} bits")
print(f"    Range=[{entropy_roi.min():.4f}, {entropy_roi.max():.4f}] bits")
print(f"    Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")

# ============================================================================
# 自动选择切片
# ============================================================================

if AUTO_SELECT_SLICES:
    print("\n自动选择切片...")
    axis_size = pred_softmax.shape[0]
    slice_scores = []
    
    for z in range(axis_size):
        mask_slice = region_mask[z] > 0
        if mask_slice.sum() > 100:
            # 使用熵的标准差作为信息量度量
            entropy_std = entropy_map[z][mask_slice].std()
            mask_ratio = mask_slice.sum() / mask_slice.size
            slice_scores.append((z, entropy_std * mask_ratio))
    
    slice_scores.sort(key=lambda x: x[1], reverse=True)
    selected_slices = []
    for z, score in slice_scores:
        if len(selected_slices) == 0 or all(abs(z - s) >= 10 for s in selected_slices):
            selected_slices.append(z)
        if len(selected_slices) >= 3:
            break
    
    MANUAL_SLICES = sorted(selected_slices)
    print(f"  选择的切片: {MANUAL_SLICES}")

print("\n✓ 数据准备完成")

In [ ]:
## 3. A+B并排可视化：概率叠加 vs 不确定性热图

# print("绘制A+B并排对比图...\n")

for slice_idx in MANUAL_SLICES:
    print(f"处理切片 {slice_idx}...")
    
    # 提取切片
    anatomy_slice = anatomy_img[slice_idx]
    top1_slice = top1_class[slice_idx]
    alpha_slice = alpha_map[slice_idx]
    mask_slice = region_mask[slice_idx]
    entropy_slice = entropy_map_masked[slice_idx]
    margin_slice = margin[slice_idx]
    
    # 创建2列图形 (A图 + B图)
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    
    # ========================================================================
    # 左图 (A): 概率叠加（Top-1 + 置信度）
    # ========================================================================
    
    # 背景：MPRAGE
    anatomy_norm = (anatomy_slice - anatomy_slice.min()) / (anatomy_slice.max() - anatomy_slice.min() + 1e-8)
    axes[0].imshow(anatomy_norm, cmap='gray', aspect='auto', interpolation='bilinear')
    
    # 叠加：Top-1类别 + 透明度
    top1_colored = cmap_freesurfer(top1_slice)
    top1_colored[..., 3] = alpha_slice
    axes[0].imshow(top1_colored, aspect='auto', interpolation='nearest')
    
    # ROI边界
    axes[0].contour(mask_slice, levels=[0.5], colors='cyan', 
                   linewidths=1.0, linestyles='--', alpha=0.5)
    
    axes[0].set_title(f'A: Probability Overlay\n'
                     f'{SUBJECT_ID} - Slice {slice_idx}\n'
                     f'Color=Top-1 ROI, Alpha=Confidence (p1-p2)',
                     fontsize=12, fontweight='bold', color='white')
    axes[0].axis('off')
    
    # ========================================================================
    # 右图 (B): 不确定性热图（熵图 + 边际差）
    # ========================================================================
    
    # 背景：边际差（灰度，归一化到[0,1]）
    margin_norm = (margin_slice - margin_slice[mask_slice > 0].min()) / \
                  (margin_slice[mask_slice > 0].max() - margin_slice[mask_slice > 0].min() + 1e-8)
    margin_norm[mask_slice == 0] = 0
    axes[1].imshow(margin_norm, cmap='gray', aspect='auto', interpolation='bilinear', 
                  vmin=0, vmax=1)
    
    # 叠加：熵图（伪彩色）
    # 使用'hot'或'jet' colormap，高熵=红色（难预测），低熵=蓝色（易预测）
    entropy_masked = np.ma.masked_where(mask_slice == 0, entropy_slice)
    
    im = axes[1].imshow(entropy_masked, cmap='hot', aspect='auto', 
                       interpolation='bilinear', alpha=0.7,
                       vmin=0, vmax=np.percentile(entropy_roi, 95))  # 95%分位数作为上限
    
    # 添加色条（熵值）
    cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label('Entropy (bits)', rotation=270, labelpad=15, color='white', fontsize=10)
    cbar.ax.tick_params(colors='white', labelsize=9)
    
    # ROI边界
    axes[1].contour(mask_slice, levels=[0.5], colors='cyan', 
                   linewidths=1.0, linestyles='--', alpha=0.5)
    
    # 标注高不确定性区域（熵 > 75%分位数）
    high_entropy_threshold = np.percentile(entropy_roi, 75)
    high_entropy_mask = (entropy_slice > high_entropy_threshold) & (mask_slice > 0)
    
    if high_entropy_mask.sum() > 0:
        # 绘制高熵区域的等值线
        axes[1].contour(entropy_slice, levels=[high_entropy_threshold], 
                       colors='yellow', linewidths=2.0, alpha=0.8)
    
    axes[1].set_title(f'B: Uncertainty Heatmap\n'
                     f'Background=Margin (p1-p2), Overlay=Entropy H(p)\n'
                     f'High Entropy (yellow) = Hard Regions',
                     fontsize=12, fontweight='bold', color='white')
    axes[1].axis('off')
    
    # ========================================================================
    # 统一设置
    # ========================================================================
    
    plt.tight_layout()
    fig.patch.set_facecolor('black')
    
    # 保存
    save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_AB_slice{slice_idx:03d}_uncertainty.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
    print(f"  ✓ 已保存: {save_path.name}")
    
    plt.show()
    plt.close()

print(f"\n✓ A+B并排对比完成！")

In [ ]:
## Figure 1: 三联图 - Reference Mode | Top-1 | Top-3 Union + 未覆盖红遮罩
# ============================================================================
# ISMRM 图像制图工程师 - Figure 1 生成
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ============================================================================
# 参数配置
# ============================================================================

SLICE_IDX = 17  # 可手动切换切片
TAU_HARD = True  # 使用硬规则（参考众数是否落入Top-3）
RED_MASK_COLOR = (220/255, 0, 0, 0.6)  # RGBA for red mask
K_TOP = 3  # Top-K

print(f"=== Figure 1 三联图生成 ===")
print(f"被试: {SUBJECT_ID}, 切片: {SLICE_IDX}")
print(f"规则: 硬规则（参考众数是否落入Top-{K_TOP}）")

# ============================================================================
# 提取切片数据
# ============================================================================

# 预测后验 (post-T)
p_pred_slice = pred_softmax[SLICE_IDX, :, :, :]  # (H, W, C=102)

# 软参考标签
p_ref_slice = gt_proba[SLICE_IDX, :, :, :]  # (H, W, C=102)

# ROI掩膜
mask_slice = region_mask[SLICE_IDX, :, :]  # (H, W)

# 解剖底图（可选）
anatomy_slice = anatomy_img[SLICE_IDX, :, :]

# ============================================================================
# 计算三个面板的内容
# ============================================================================

# (a) 参考众数（Reference mode）: argmax of soft reference
ref_mode = np.argmax(p_ref_slice, axis=-1)  # (H, W)

# (b) Top-1 预测
pred_top1 = np.argmax(p_pred_slice, axis=-1)  # (H, W)

# (c) Top-3 union + 红遮罩
# 获取 Top-3 classes (每个像素)
top3_indices = np.argsort(p_pred_slice, axis=-1)[..., -K_TOP:]  # (H, W, 3)

# 硬规则：检查 ref_mode 是否在 pred_top3 中
covered_mask = np.zeros(ref_mode.shape, dtype=bool)  # (H, W)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:  # 只在ROI内计算
            ref_class = ref_mode[i, j]
            pred_top3_classes = top3_indices[i, j, :]
            covered_mask[i, j] = ref_class in pred_top3_classes

# 红遮罩：ref_mode 不在 Top-3 中
red_mask = (~covered_mask) & (mask_slice > 0)  # (H, W)

# ============================================================================
# 统计信息（校验输出）
# ============================================================================

roi_pixels = mask_slice > 0
total_roi = roi_pixels.sum()
red_pixels = red_mask.sum()
red_ratio = 100 * red_pixels / total_roi if total_roi > 0 else 0

print(f"\n=== 统计信息 ===")
print(f"ROI 总像素数: {total_roi}")
print(f"红遮罩像素数: {red_pixels}")
print(f"红遮罩比例: {red_ratio:.2f}%")
print(f"Top-{K_TOP} 覆盖率: {100 - red_ratio:.2f}%")

# 软覆盖分数分布（额外信息，用于对比）
soft_coverage = np.zeros(ref_mode.shape)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:
            pred_top3_classes = top3_indices[i, j, :]
            soft_coverage[i, j] = p_ref_slice[i, j, pred_top3_classes].sum()

soft_cov_roi = soft_coverage[roi_pixels]
print(f"\n软覆盖分数（额外信息）:")
print(f"  Mean={soft_cov_roi.mean():.4f}, Median={np.median(soft_cov_roi):.4f}")
print(f"  Range=[{soft_cov_roi.min():.4f}, {soft_cov_roi.max():.4f}]")

# ============================================================================
# 绘制三联图
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(36, 12))

# 通用设置函数
def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    """
    绘制分类图（统一LUT，无透明度）
    
    Parameters:
    - ax: matplotlib axis
    - label_map: (H, W) 整数标签
    - mask: (H, W) ROI掩膜
    - title: 标题
    - show_red_mask: 是否叠加红遮罩
    - red_mask_data: (H, W) 红遮罩布尔数组
    """
    # 背景黑色
    label_rgb = np.zeros((*label_map.shape, 3))
    
    # 应用FreeSurfer LUT
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            color = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
            label_rgb[class_mask] = color
    
    # 显示分类图（无透明度）
    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')
    
    # 叠加红遮罩（如果需要）
    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')
    
    # ROI边界（青色虚线）
    ax.contour(mask, levels=[0.5], colors='cyan', 
               linewidths=1.0, linestyles='--', alpha=0.5)
    
    ax.set_title(title, fontsize=14, fontweight='bold', color='white', pad=10)
    ax.axis('off')

# ========================================================================
# (a) Reference Mode
# ========================================================================
plot_categorical_map(
    axes[0], 
    ref_mode, 
    mask_slice,
    f'(a) Reference Mode\n(argmax of soft reference)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (b) Top-1 Prediction
# ========================================================================
plot_categorical_map(
    axes[1], 
    pred_top1, 
    mask_slice,
    f'(b) Top-1 Prediction\n(argmax of p_pred, post-T)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (c) Top-3 Union + 红遮罩
# ========================================================================
plot_categorical_map(
    axes[2], 
    ref_mode,  # 显示参考颜色作为底色
    mask_slice,
    f'(c) Top-{K_TOP} Union + Uncovered Mask\n' +
    f'Red = Ref mode NOT in pred Top-{K_TOP}\n' +
    f'Coverage: {100-red_ratio:.1f}%, Uncovered: {red_ratio:.1f}%',
    show_red_mask=True,
    red_mask_data=red_mask
)

# ========================================================================
# 图例（可选，显示几个主要脑区颜色）
# ========================================================================

# 选择几个代表性标签
legend_labels = []
for class_idx in [0, 2, 7, 10, 41, 42]:  # 示例：背景+几个主要脑区
    if class_idx in LABEL_NAMES:
        legend_labels.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx], 
                  edgecolor='white', label=LABEL_NAMES[class_idx])
        )

# 红遮罩图例
legend_labels.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

# 添加图例到右侧
fig.legend(handles=legend_labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
           frameon=True, facecolor='black', edgecolor='white', 
           fontsize=10, labelcolor='white')

# ========================================================================
# 统一设置
# ========================================================================

plt.tight_layout()
fig.patch.set_facecolor('black')

# 添加总标题
fig.suptitle(f'Figure 1: Reference Mode | Top-1 Prediction | Top-{K_TOP} Union with Uncovered Regions\n' +
             f'Subject: {SUBJECT_ID}, Slice: {SLICE_IDX}, {SPLIT_TYPE.upper()} set',
             fontsize=16, fontweight='bold', color='white', y=0.98)

plt.show()

# ============================================================================
# 保存统计数据（CSV）- 可选
# ============================================================================

# 软覆盖直方图（10 bins）
hist, bin_edges = np.histogram(soft_cov_roi, bins=10, range=(0, 1))
hist_data = {
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': hist,
    'percentage': 100 * hist / total_roi
}

print(f"\n=== 软覆盖分数直方图 (10 bins) ===")
for i in range(10):
    print(f"  [{hist_data['bin_left'][i]:.2f}, {hist_data['bin_right'][i]:.2f}): " +
          f"{hist_data['count'][i]} pixels ({hist_data['percentage'][i]:.2f}%)")

print(f"\n✓ Figure 1 生成完成！")

## Figure 1 图注

**Fig. 1.** Reference mode (argmax of the **soft** reference), Top-1 predictions, and the Top-3 union. A voxel is highlighted in red if the reference mode (argmax of soft reference) is **NOT** in the predicted Top-3 classes. Same FreeSurfer LUT across all panels; no transparency on categorical maps; reference mode is for display only; evaluation uses soft references. Boundary misses cluster at cortical–white matter interfaces. Values are from **post-T** predictions.

---

### 代码说明

**核心功能**：
- **(a) Reference Mode**: 显示软参考标签的众数（argmax），仅用于可视化对比
- **(b) Top-1 Prediction**: 显示模型预测的最高概率类别（post-T）
- **(c) Top-3 Union + 未覆盖红遮罩**: 
  - **硬规则**：检查参考众数是否落入预测的 Top-3 类别中
  - **红色遮罩**：参考众数未被 Top-3 覆盖的像素
  - 有助于识别模型的"困难区域"（通常在边界）

**可调参数**：
- `SLICE_IDX = 17`: 切片索引（可手动修改）
- `K_TOP = 3`: Top-K 值（默认3，可改为5、10等）
- `RED_MASK_COLOR`: 红遮罩的 RGBA 颜色

**输出信息**：
1. ROI总像素数、红遮罩像素数、红遮罩比例
2. Top-K 覆盖率统计
3. 软覆盖分数分布（10 bins 直方图）

**用途**：
- ISMRM 论文投稿图像
- 模型性能可视化
- 错误分析（红色区域集中在哪些解剖结构）

---

### 如何使用

1. **切换切片**：修改 `SLICE_IDX` 变量
2. **切换 Top-K**：修改 `K_TOP` 变量（例如改为 5 查看 Top-5 覆盖）
3. **导出高分辨率图像**（可选）：
   ```python
   # 在 plt.show() 之前添加：
   plt.savefig('Fig1_topK_union.png', dpi=600, bbox_inches='tight', facecolor='black')
   plt.savefig('Fig1_topK_union.pdf', bbox_inches='tight', facecolor='black')
   ```

---